# Yantra — resumable DTSA agentic-training pipeline (Colab T4)

Self-contained pipeline that trains **Yantra** (a <=1B agentic tool-use model on
MiniCPM5-1B) and compares it, in-pipeline, against the
`MiniCPM5-1B-Agentic-Tooluse` baseline on the same 300-case ToolACE eval set.

**Resumability:** every stage writes a marker under `RUN_DIR` and skips if the
marker exists. Long training stages also save periodic LoRA checkpoints and
resume from the latest one, so a crash / Colab timeout just means *Run All*
again — completed work is not repeated.

**Persistence:** artifacts live in `RUN_DIR`:
- Google Drive `/content/drive/MyDrive/yantra_run` if Drive is mounted,
- else `./yantra_run` (local fallback, used when testing off-Colab).

**How to run:** `Runtime → Run all`. First run installs deps + mounts Drive
(~few min). Training stages need a GPU (T4 free tier is fine for 1B QLoRA).
They are guarded — on a non-CUDA machine they print a skip and the rest of the
notebook (data + baseline eval + resume logic) still runs.

Stages: 0 setup · 1 data · 2 baseline eval · 3 DTSA SFT · 4 EG-OPD self-distill
· 5 RTE SFT · 6 GGUF→Yantra eval · 7 summary · 8 HF upload.

**Reliability rules baked in (each fixed a real failure):**
- No pinned xformers/torch versions — pins break when Colab bumps Python.
- All multi-GB writes go to local disk first; Drive mirrors are best-effort
  (Drive FUSE throws `[Errno 5]` on large writes).
- Serving uses Unsloth's `llama-server` binary, always on a local-disk copy of
  the GGUF (FUSE reads time out), after killing stale servers on :8000.
- Stage 6 reuses an existing Q4 GGUF if present, so a crash during eval does
  not re-run the ~15 min export.

In [ ]:
# @title Stage 0 — setup: deps, Drive, resume helpers
import os, sys, json, re, math, shutil, subprocess, time, threading
from pathlib import Path
from huggingface_hub import snapshot_download

# ---- Google Drive persistence (falls back to local) ----
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/yantra_run")
except Exception:
    RUN_DIR = Path("./yantra_run")
RUN_DIR.mkdir(parents=True, exist_ok=True)
MARKERS = RUN_DIR / "markers"
MARKERS.mkdir(parents=True, exist_ok=True)
ART = RUN_DIR / "artifacts"
ART.mkdir(parents=True, exist_ok=True)
print("RUN_DIR =", RUN_DIR)

def marker(name): return (MARKERS / name).exists()
def mark(name):
    (MARKERS / name).write_text(time.strftime("%Y-%m-%dT%H:%M:%S"))
    print(f"[marker] {name} done")
def latest_ckpt(out_dir):
    out_dir = Path(out_dir)
    if not out_dir.exists(): return None
    cs = sorted(
        [p for p in out_dir.iterdir() if p.is_dir() and p.name.startswith("checkpoint-") and (p / "trainer_state.json").exists()],
        key=lambda p: int(re.search(r"(\d+)", p.name).group(1)))
    return cs[-1] if cs else None

IS_COLAB = any(k.startswith("COLAB_") for k in os.environ)
BASE_MODEL = "openbmb/MiniCPM5-1B"
BASELINE_REPO = "ewinregirgojr/MiniCPM5-1B-Agentic-Tooluse-GGUF"
BASELINE_GGUF = "MiniCPM5-1B-Agentic-Tooluse-Nemotron-DPO.Q4_K_M.gguf"

# Base model download helper. Stage 3 calls base_model_dir() in the FOREGROUND so
# progress is visible and LFS stalls fail fast (no silent background-thread hang).
# STRATEGY: pull to the LOCAL Colab disk (fast SSD), because writing ~2.2 GB through
# Google Drive's FUSE layer is extremely slow and can appear to hang. If a valid copy
# already exists on Drive, reuse it (skip). After a local download, mirror to Drive in
# the background so future runs skip. Drive FUSE has no symlink support, so always copy
# real files (local_dir_use_symlinks=False). is_file() rejects broken symlinks.
_LOCAL_BASE = Path("/content/MiniCPM5-1B")

def _shards_ok(d):
    idx = d / "model.safetensors.index.json"
    if not idx.exists():
        return False
    try:
        shards = set(json.load(open(idx))["weight_map"].values())
        return all((d / s).is_file() for s in shards)  # is_file rejects broken symlinks
    except Exception:
        return False

def base_model_dir():
    # 1) reuse a real Drive copy if present (fast skip on later runs)
    if IS_COLAB and _shards_ok(ART / "base_model"):
        print("base model already on Drive; reusing")
        return ART / "base_model"
    # 2) otherwise download to local disk (fast), then mirror to Drive in background
    d = _LOCAL_BASE if IS_COLAB else (ART / "base_model")
    if not _shards_ok(d):
        print("downloading base model", BASE_MODEL, "to", d, "(local, fast)")
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # fast LFS-capable CDN (local disk target)
        snapshot_download(BASE_MODEL, local_dir=str(d), local_dir_use_symlinks=False)
    if IS_COLAB:
        def _mirror():
            try:
                import shutil
                dst = ART / "base_model"; dst.mkdir(parents=True, exist_ok=True)
                for f in d.iterdir():
                    if f.is_file():
                        shutil.copy2(f, dst / f.name)
                print("base model mirrored to Drive")
            except Exception as e:
                print("base model Drive mirror skipped:", e)
        threading.Thread(target=_mirror, daemon=True).start()
    return d
# NOTE: the base model is now downloaded in the FOREGROUND inside stage3() (via
# base_model_dir()) so progress is visible and LFS stalls fail fast, instead of
# hanging silently in a background thread.

# deps: install only if missing (Colab already has CUDA torch; local test gets CPU torch)
def need(mod, cmd):
    try:
        __import__(mod); return
    except Exception:
        print("installing", mod, "..."); subprocess.run(cmd, shell=True, check=True)
need("datasets", "pip install -q datasets huggingface_hub")
need("torch", "pip install -q torch")

# Unsloth install. RULE: never pin xformers/torch versions here — Colab bumps
# its Python/torch regularly and any pin eventually has no wheel (that exact
# crash happened when a runtime jumped to py3.13). Plain `pip install unsloth`
# resolves a compatible stack on whatever runtime is booted. Gate on
# IMPORTABILITY (not a marker): the marker lives on Drive and survives restarts,
# so a fresh runtime must still reinstall. Install failures are NON-FATAL.
def _sh(cmd):
    return subprocess.run(cmd, shell=True).returncode == 0

try:
    import unsloth  # noqa: F401
    _unsloth_ok = True
except Exception:
    _unsloth_ok = False
if IS_COLAB and not _unsloth_ok:
    print("installing unsloth ...")
    if not _sh("pip install -q --upgrade unsloth unsloth_zoo"):
        print("PyPI unsloth failed; trying git colab-new build ...")
        _sh('pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
    _sh("pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer")
elif not IS_COLAB:
    print("local run: skipping unsloth install (stubbed later)")

# NOTE: no llama-cpp-python preinstall here. Serving prefers Unsloth's llama.cpp
# `llama-server` binary (installed at /root/.unsloth/llama.cpp during GGUF export);
# ensure_server() installs the python fallback lazily and best-effort if ever needed.

import torch
print("cuda available:", torch.cuda.is_available())
HAS_CUDA = torch.cuda.is_available()


In [ ]:
# @title Shared utils (ToolACE extractor, DTSA, verifier, parsers, eval, PAS)
import json, re, random
from typing import Any

# ---------- ToolACE live ShareGPT extractor ----------
TOOLS_MARKER = "Should you decide to return the function call"
def extract_tools(system):
    if not system: return []
    m = re.search(r"(\[\s*\{.*\}\])\s*[.\s]*" + re.escape(TOOLS_MARKER), system, re.S)
    if not m: return []
    try: return json.loads(m.group(1).strip())
    except json.JSONDecodeError: return []
def _split_top(s):
    out, depth, buf, quote = [], 0, "", None
    for ch in s:
        if quote:
            buf += ch
            if ch == quote: quote = None
            continue
        if ch in ("'", '"'): quote = ch; buf += ch
        elif ch in "[{": depth += 1; buf += ch
        elif ch in "]}": depth -= 1; buf += ch
        elif ch == "," and depth == 0: out.append(buf); buf = ""
        else: buf += ch
    if buf.strip(): out.append(buf)
    return out
def _parse_value(v):
    v = v.strip()
    if (v.startswith('"') and v.endswith('"')) or (v.startswith("'") and v.endswith("'")): return v[1:-1]
    low = v.lower()
    if low in ("true","false","null"): return {"true":True,"false":False,"null":None}[low]
    try: return int(v)
    except ValueError: pass
    try: return float(v)
    except ValueError: pass
    try: return json.loads(v)
    except Exception: return v
def parse_kwargs(s):
    a = {}
    if not s.strip(): return a
    for part in _split_top(s):
        if "=" not in part: continue
        k, val = part.split("=", 1); a[k.strip()] = _parse_value(val)
    return a
def parse_calls(text):
    res = []
    for seg in re.findall(r"\[([^\[\]]*)\]", text):
        for m in re.finditer(r"([A-Za-z0-9_ ]+?)\s*\(([^\(\)]*)\)", seg):
            n = m.group(1).strip()
            if n: res.append({"name": n, "arguments": parse_kwargs(m.group(2))})
    return res
def extract_example(ex):
    conv = ex.get("conversations") or []
    tools = extract_tools(ex.get("system", ""))
    if not tools: return None
    ut = next((c for c in conv if c.get("from") == "user"), None)
    if not ut: return None
    gold = []
    for c in conv:
        if c.get("from") == "assistant":
            calls = parse_calls(c["value"])
            if calls: gold = calls; break
    if not gold: return None
    return {"query": ut["value"], "tools": [{"function": t} for t in tools], "gold": gold}

# ---------- DTSA (Decoupled Tool Selection / Argument generation) ----------
ACTION_END = "<action_end/>"
DTSA_BIND = re.compile(r'<bind\s+tool="([^"]+)"\s*/>', re.S)
DTSA_ARGS = re.compile(r"<args>(.*?)</args>", re.S)
DTSA_PARAM = re.compile(r'<param\s+name="([^"]+)"\s*>(.*?)</param>', re.S)
def to_dtsa(name, arguments):
    lines = ['<bind tool="%s"/>' % name, "<args>"]
    for k, v in arguments.items():
        v = "" if v is None else str(v)
        v = v.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        lines.append('  <param name="%s">%s</param>' % (k, v))
    lines += ["</args>", ACTION_END]
    return "\n".join(lines)
def build_user_prompt(query, tools):
    tj = json.dumps([t.get("function", t) for t in tools], ensure_ascii=False, indent=2)
    return f"<user>{query}</user>\n<tools>{tj}</tools>\n<calls>"
# DTSA design: the runtime/router binds <bind tool="..."/>; the LM learns to
# emit ONLY the <args>...</args><action_end/> block. So training prompts carry
# the bind and completions are args-only (consistent with eval in Stage 6).
def bind_prefix(name): return f'<bind tool="{name}"/>\n'
def dtsa_args_block(name, args):
    return "\n".join(to_dtsa(name, args).splitlines()[1:])  # drop the <bind .../> line
def strip_bind(text):
    m = DTSA_BIND.search(text)
    return text[m.end():] if m else text
def parse_args_block(text):
    am = DTSA_ARGS.search(text)
    if not am: return {}, False
    args = {pm.group(1): _unesc(pm.group(2).strip()) for pm in DTSA_PARAM.finditer(am.group(1))}
    ends = list(re.finditer(re.escape(ACTION_END), text))
    stopped = bool(ends) and text[ends[-1].end():].strip() == ""
    return args, stopped

# ---------- Verifier (schema + optional mock exec) for EG-OPD reward ----------
class ToolVerifier:
    def __init__(self, registry=None): self.registry = registry or {}
    def validate_schema(self, args, schema):
        props = schema.get("properties", {}); req = schema.get("required", [])
        for r in req:
            if r not in args or args[r] in (None, ""): return False, f"Missing:{r}"
        for k, v in args.items():
            spec = props.get(k);
            if spec is None: continue
            t = spec.get("type")
            if t == "integer" and not isinstance(v, int): return False, f"Type:{k}"
            if t == "number" and not isinstance(v, (int, float)): return False, f"Type:{k}"
            if t == "string" and not isinstance(v, str): return False, f"Type:{k}"
            if "enum" in spec and v not in spec["enum"]: return False, f"Enum:{k}"
        return True, ""
    def reward(self, name, args, schema):
        ok, err = self.validate_schema(args, schema)
        if not ok: return 0.0, err
        fn = self.registry.get(name)
        if fn is None: return 0.5, "schema-valid"
        try:
            fn(**args); return 1.0, "executed"
        except Exception as e: return 0.5, f"exec-error:{e}"

# ---------- Parsers ----------
def _unesc(v): return v.replace("&amp;","&").replace("&lt;","<").replace("&gt;",">")
def parse_dtsa(text):
    # Grade the FIRST call (the runtime-bound one). The model is trained to emit
    # args-only; any later <bind> it invents is post-answer rambling and must not
    # hijack scoring (binds[-1] used to grade the hallucinated follow-up call).
    binds = list(DTSA_BIND.finditer(text))
    if not binds: return None
    lb = binds[0]; tool = lb.group(1)
    seg_end = binds[1].start() if len(binds) > 1 else len(text)
    am = DTSA_ARGS.search(text[lb.end():seg_end])
    args = {}
    if am:
        for pm in DTSA_PARAM.finditer(am.group(1)): args[pm.group(1)] = _unesc(pm.group(2).strip())
    ends = list(re.finditer(re.escape(ACTION_END), text))
    stopped = bool(ends) and text[ends[-1].end():].strip() == ""
    return {"tool": tool, "args": args, "stopped_clean": stopped}
def parse_legacy(text):
    m = re.search(r'<function\s+name="([^"]+)"\s*>(.*?)</function>', text, re.S)
    if not m: return None
    tool = m.group(1); args = {pm.group(1): pm.group(2).strip()
        for pm in re.finditer(r'<param\s+name="([^"]+)"\s*>(.*?)</param>', m.group(2), re.S)}
    tail = text[m.end():].strip()
    return {"tool": tool, "args": args, "stopped_clean": tail == ""}
def parse_lenient(text):
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    m = parse_legacy(t)
    if m: return m
    ms = list(re.finditer(r'name="([^"]+)"\s*>', t))
    if not ms: return None
    tool = ms[0].group(1); args = {}
    for i, mm in enumerate(ms[1:], 1):
        start = mm.end(); end = ms[i+1].start() if i+1 < len(ms) else len(t)
        val = t[start:end].lstrip(">").strip()
        if val: args[mm.group(1)] = val
    return {"tool": tool, "args": args, "stopped_clean": bool(re.search(r"</function>|"+re.escape(ACTION_END), t))}

# ---------- Eval metrics ----------
def _norm(a):
    if isinstance(a, str):
        s = a.strip()
        if s.lower() in ("true","false"): return s.lower()=="true"
        try: return int(s) if "." not in s else float(s)
        except ValueError: return s
    return a
def avail_names(tools): return {t.get("function", t).get("name") for t in tools}
def evaluate_case(text, case, mode):
    tools = case["tools"]; gold = case["gold"]; gold0 = gold[0]
    parsed = parse_dtsa(text) if mode == "dtsa" else parse_lenient(text)
    parseable = parsed is not None
    if not parsed:
        return {"parseable":0,"valid_name":0,"expected_name":0,"exact_args":0,"arg_key_overlap":0,"stopped_cleanly":0}
    valid = parsed["tool"] in avail_names(tools)
    expected = parsed["tool"] == gold0["name"]
    gk = set(gold0["arguments"].keys()); pk = set(parsed["args"].keys())
    overlap = len(gk & pk)/len(gk) if gk else 1.0
    exact = (set(parsed["args"].keys()) == gk) and all(_norm(gold0["arguments"][k]) == _norm(parsed["args"][k]) for k in gk) if gk else bool(parsed)
    return {"parseable":1,"valid_name":int(valid),"expected_name":int(expected),
            "exact_args":int(exact),"arg_key_overlap":round(overlap,4),"stopped_cleanly":int(parsed["stopped_clean"])}

def pas(summary, recovery=0.0, multiturn=0.0):
    comps = [summary.get(k,0.0) for k in ("parseable","valid_name","expected_name","exact_args","arg_key_overlap","stopped_cleanly")] + [recovery, multiturn]
    return round(sum(comps)/len(comps), 4)
print("utils loaded")


In [ ]:
# @title Stage 1 — data: build DTSA SFT set, RTE set, 300-case eval set
from datasets import load_dataset

def build_sets():
    ds = load_dataset("Team-ACE/ToolACE", split="train")
    dtsa, rte, seen = [], [], []
    err_templates = [
        ("404: unknown {p} '{v}'", "typo"), ("ValidationError: {p} must be one of [enum]; got '{v}'", "enum"),
        ("TypeError: {p} expected int, got str '{v}'", "type"), ("MissingRequiredArgument: {p} is required", "missing"),
        ("EmptyResult: no data for {p}='{v}'", "empty")]
    rng = random.Random(7)
    for ex in ds:
        e = extract_example(ex)
        if not e: continue
        gold = e["gold"][0]
        # DTSA SFT record (one assistant turn per gold call)
        dtsa.append({"query": e["query"], "tools": e["tools"], "gold": e["gold"],
                     "prompt": build_user_prompt(e["query"], e["tools"]),
                     "completion": to_dtsa(gold["name"], gold["arguments"])})
        # RTE: corrupt one arg -> error -> corrected
        ks = list(gold["arguments"].keys())
        if ks:
            k = rng.choice(ks); v = gold["arguments"][k]; kind = rng.choice(err_templates)
            if kind[1] == "missing":
                corrupted = {kk: vv for kk, vv in gold["arguments"].items() if kk != k}
                err = kind[0].format(p=k, v="")
            elif kind[1] == "typo":
                bad = (str(v) + "x") if v not in ("", None) else "zzz"
                corrupted = {**gold["arguments"], k: bad}; err = kind[0].format(p=k, v=bad)
            else:
                corrupted = {**gold["arguments"], k: v}; err = kind[0].format(p=k, v=v)
            rte.append({"query": e["query"], "tools": e["tools"],
                        "failed": to_dtsa(gold["name"], corrupted),
                        "error": err, "corrected": to_dtsa(gold["name"], gold["arguments"])})
    # 300-case eval: single-call, deterministic seed 42
    single = [e for e in (extract_example(x) for x in ds) if e and len(e["gold"]) == 1]
    rng2 = random.Random(42); rng2.shuffle(single)
    eval_set = [{"id": i, "query": r["query"], "tools": r["tools"],
                 "relevant_tools": [], "gold": r["gold"]} for i, r in enumerate(single[:300])]
    out = {
        "dtsa_sft.jsonl": dtsa, "rte.jsonl": rte, "toolace_300.jsonl": eval_set,
    }
    for name, rows in out.items():
        with open(ART / name, "w") as f:
            for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"dtsa={len(dtsa)} rte={len(rte)} eval={len(eval_set)}")

if not marker("stage1_data"):
    build_sets(); mark("stage1_data")
else:
    print("stage1_data: SKIP (already done)")


In [ ]:
# @title Stage 2 — baseline eval (served via llama.cpp; tolerant parser)
import subprocess, urllib.request, threading, shutil, os, glob

def _find_llama_server():
    import glob as _g, shutil as _s
    cands = (_g.glob("/root/.unsloth/llama.cpp/llama-server") +
             _g.glob("/usr/local/bin/llama-server"))
    w = _s.which("llama-server")
    if w: cands.append(w)
    return next((c for c in cands if os.path.exists(c)), None)

def ensure_server(gguf_path, port=8000):
    # Kill any stale server first — a leftover on :8000 would pass the health
    # check below while silently serving a DIFFERENT model.
    subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
    subprocess.run(["pkill", "-f", "llama_cpp.server"], capture_output=True)
    time.sleep(2)
    # Always serve from LOCAL disk: reading a 656MB GGUF through Drive FUSE
    # makes even the health check time out. Size-check guards partial copies.
    serve_path = gguf_path
    if IS_COLAB:
        cand = Path("/content") / gguf_path.name
        if not cand.exists() or cand.stat().st_size != gguf_path.stat().st_size:
            print(f"copying GGUF to local disk ({gguf_path.stat().st_size//1024//1024}MB) ...")
            shutil.copy2(str(gguf_path), str(cand))
        serve_path = cand
    log = open(RUN_DIR/"server.log","w")
    binp = _find_llama_server()
    if binp:
        print("serving with", binp)
        p = subprocess.Popen([str(binp), "-m", str(serve_path),
                              "--port", str(port), "-c", "2048"],
                             stdout=log, stderr=subprocess.STDOUT)
    else:
        # Fallback: python llama server, installed lazily + best-effort.
        try:
            import llama_cpp.server  # noqa: F401
        except Exception:
            print("installing llama-cpp-python[server] (one-time fallback) ...")
            subprocess.run("pip install -q 'llama-cpp-python[server]' --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121", shell=True)
        p = subprocess.Popen([sys.executable, "-m", "llama_cpp.server", "--model", str(serve_path),
                              "--port", str(port), "--n_gpu_layers", "-1"],
                             stdout=log, stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2); return p
        except Exception: time.sleep(2)
    try:
        out = (RUN_DIR/"server.log").read_text()
    except Exception:
        out = ""
    raise RuntimeError("server did not start\n--- server.log tail ---\n" + "\n".join(out.splitlines()[-25:]))

def _gguf_url():
    return f"https://huggingface.co/{BASELINE_REPO}/resolve/main/{BASELINE_GGUF}"

def eval_baseline():
    # Baseline GGUF is an HF *LFS* file; the hub/LFS CDN is often blocked or stalls
    # from Colab (observed: 30 min at 0%). Strategy:
    #   1) reuse a copy already on Drive (drop the .gguf there to skip HF entirely),
    #   2) else pull via curl with timeouts (fails fast instead of hanging 30 min).
    # Download to LOCAL disk (fast serving + avoids Drive FUSE); mirror to Drive.
    import subprocess, shutil
    local_dir = Path("/content") if IS_COLAB else ART
    gguf_path = local_dir / BASELINE_GGUF
    # 1) reuse a Drive copy if present
    if IS_COLAB and (ART / BASELINE_GGUF).is_file():
        print("baseline GGUF found on Drive; reusing")
        gguf_path = ART / BASELINE_GGUF
    # 2) download via curl (LFS-aware; connect-timeout fails fast if blocked)
    if not gguf_path.is_file():
        print("downloading baseline GGUF via curl ...")
        out = str(gguf_path)
        rc = subprocess.run(
            ["curl", "-L", "--fail", "--retry", "3", "--connect-timeout", "25",
             "--max-time", "1200", "-C", "-", "-o", out, _gguf_url()],
            capture_output=True, text=True).returncode
        if rc != 0 or not gguf_path.is_file() or gguf_path.stat().st_size == 0:
            print(
                "WARNING: baseline GGUF could not be downloaded (HF LFS is often blocked from "
                "Colab). Training WILL continue without baseline comparison.\n"
                f"  To get the baseline later, download '{BASELINE_GGUF}' and upload to:\n"
                f"    {ART / BASELINE_GGUF}\n"
                f"  Then delete the marker at {MARKERS}/stage2_baseline and re-run Stage 2.")
            (ART/"baseline_results.json").write_text('{"pas": "N/A (no GGUF)", "note": "baseline GGUF not available; upload manually to compare"}')
            return
        print("gguf downloaded to", gguf_path)
    # mirror to Drive in background for future reuse (skip if already on Drive)
    if IS_COLAB and not (ART / BASELINE_GGUF).is_file():
        def _mirror():
            try:
                (ART / BASELINE_GGUF).write_bytes(gguf_path.read_bytes())
                print("baseline GGUF mirrored to Drive")
            except Exception as e:
                print("baseline GGUF Drive mirror skipped:", e)
        threading.Thread(target=_mirror, daemon=True).start()
    gguf = gguf_path
    server = ensure_server(gguf)
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="x", timeout=120.0)
    cases = [json.loads(l) for l in open(ART/"toolace_300.jsonl") if l.strip()]
    per = []
    for i, case in enumerate(cases):
        prompt = build_user_prompt(case["query"], case["tools"])
        r = client.chat.completions.create(model="baseline", messages=[{"role":"user","content":prompt}],
                                            temperature=0, max_tokens=256)
        m = evaluate_case(r.choices[0].message.content or "", case, mode="legacy")
        per.append({"id": i, **m})
    agg = {}
    for m in per:
        for k, v in m.items():
            if k == "id": continue
            agg[k] = agg.get(k, 0.0) + v
    n = len(cases); summary = {k: round(v/n, 4) for k, v in agg.items()}
    score = pas(summary)
    res = {"summary": summary, "pas": score, "per_case": per}
    (ART/"baseline_results.json").write_text(json.dumps(res, indent=2))
    if server: server.terminate()
    print("BASELINE PAS =", score); print(json.dumps(summary, indent=2))
    return res

r = json.loads((ART/"baseline_results.json").read_text()) if (ART/"baseline_results.json").exists() else {"pas": "?"}
if not marker("stage2_baseline") or (r.get("pas") == "N/A (no GGUF)" and (ART/BASELINE_GGUF).is_file()):
    if (ART/BASELINE_GGUF).is_file():
        print("baseline GGUF now on Drive; running baseline eval ...")
    eval_baseline(); mark("stage2_baseline")
else:
    print("stage2_baseline: SKIP (PAS =", r.get("pas", "?"), ")")


In [ ]:
# @title Stage 3 — DTSA SFT (unsloth QLoRA)  [CUDA required]
def stage3():
    from unsloth import FastLanguageModel
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from datasets import Dataset
    max_seq = 2048
    # Download (or reuse) the base model in the FOREGROUND so progress is visible
    # and any LFS stall fails fast instead of hanging silently in a background thread.
    base_dir = base_model_dir()
    model, tokenizer = FastLanguageModel.from_pretrained(str(base_dir), max_seq_length=max_seq, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(model, r=64, lora_alpha=128, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth")
    rows = [json.loads(l) for l in open(ART/"dtsa_sft.jsonl") if l.strip()]
    def build_text(r):
        name = r["gold"][0]["name"]
        prompt = f"<user>{r['query']}</user>\n<tools>{json.dumps([t.get('function',t) for t in r['tools']], ensure_ascii=False)}</tools>\n<calls>" + bind_prefix(name)
        return prompt + dtsa_args_block(name, r["gold"][0]["arguments"]) + "\n"
    ds = Dataset.from_list([{"text": build_text(r)} for r in rows])  # simple string column
    out = ART/"stage3_dtsa_sft"
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
        formatting_func=lambda x: x["text"],
        args=TrainingArguments(per_device_train_batch_size=4, gradient_accumulation_steps=4,
            warmup_ratio=0.03, num_train_epochs=2, learning_rate=1e-4, fp16=True,
            logging_steps=25, save_strategy="no", output_dir=str(out)))
    trainer.train()
    model.save_pretrained(str(out/"adapters")); tokenizer.save_pretrained(str(out/"adapters"))
    print("stage3 done ->", out/"adapters")

if not marker("stage3_dtsa"):
    if not HAS_CUDA:
        print("stage3_dtsa: SKIP (no CUDA — run on Colab T4). Marking skipped-not-done.")
    else:
        stage3(); mark("stage3_dtsa")
else:
    print("stage3_dtsa: SKIP (already done)")


In [ ]:
# @title Stage 4 — EG-OPD verifier-weighted self-distillation (DPO)  [CUDA required]
def stage4():
    from unsloth import FastLanguageModel
    from trl import DPOTrainer, DPOConfig
    from datasets import Dataset
    from transformers import TrainerCallback
    import warnings, logging, shutil
    warnings.filterwarnings("ignore")
    logging.getLogger("transformers").setLevel(logging.ERROR)
    out = ART/"stage4_egopd"
    out.mkdir(parents=True, exist_ok=True)
    # Resume training from a mid-training checkpoint if one exists
    ckpts = sorted(out.glob("ckpt_step*"), key=lambda p: int(p.name.replace("ckpt_step","")))
    if ckpts:
        latest = ckpts[-1]
        print(f"resuming DPO training from {latest.name}")
        model, tokenizer = FastLanguageModel.from_pretrained(str(latest), max_seq_length=2048)
        model.generation_config.max_length = None
    else:
        base = ART/"stage3_dtsa_sft"/"adapters"
        model, tokenizer = FastLanguageModel.from_pretrained(str(base), max_seq_length=2048)
        model.generation_config.max_length = None
        model = FastLanguageModel.get_peft_model(model, r=64, lora_alpha=128, lora_dropout=0.05,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    class SaveCB(TrainerCallback):
        def __init__(self, d): self.d = d
        def on_step_end(self, args, state, control, **kw):
            if state.global_step and state.global_step % 100 == 0:
                m = kw.get("model")
                if m is not None:
                    p = self.d / f"ckpt_step{state.global_step}"
                    try:
                        m.save_pretrained(str(p)); tokenizer.save_pretrained(str(p))
                        for o in sorted(self.d.glob("ckpt_step*"), key=lambda x: int(x.name.replace("ckpt_step","")))[:-2]:
                            shutil.rmtree(str(o), ignore_errors=True)
                        print(f"  checkpoint saved: {p.name}", flush=True)
                    except Exception as e:
                        print("  checkpoint save failed:", e)
    verifier = ToolVerifier()
    rows = [json.loads(l) for l in open(ART/"dtsa_sft.jsonl") if l.strip()]
    pairs_file = out/"pairs.jsonl"
    # Resume: load all existing pairs; track row indices already done
    pairs = []
    seen = set()
    if pairs_file.exists():
        by_idx = {}
        for l in pairs_file.open():
            if l.strip():
                p = json.loads(l)
                by_idx[p["row_idx"]] = p
        seen = set(by_idx); pairs = list(by_idx.values())
        print(f"resume: {len(seen)} pairs loaded, skipping their rows")
    def gen(model, tok, prompt, max_new=128):
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False, max_length=None)
        return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
    for i, r in enumerate(rows):
        if i in seen:
            continue
        name = r["gold"][0]["name"]
        prompt = f"<user>{r['query']}</user>\n<tools>{json.dumps([t.get('function',t) for t in r['tools']], ensure_ascii=False)}</tools>\n<calls>" + bind_prefix(name)
        comp = strip_bind(gen(model, tokenizer, prompt))
        args, stopped = parse_args_block(comp)
        schema = next((t["function"].get("parameters",{}) for t in r["tools"] if t["function"].get("name")==name), {})
        reward, _ = verifier.reward(name, args, schema)
        if reward >= 0.5 and stopped:
            reward = min(reward + 0.3, 1.0)  # bonus for clean termination
        chosen = dtsa_args_block(name, r["gold"][0]["arguments"])
        if reward >= 0.5:
            pairs.append({"row_idx": i, "prompt": prompt, "chosen": chosen, "rejected": comp})
        # save ALL pairs every 50 rows (overwrite = always deduped)
        if i > 0 and i % 50 == 0:
            print(f"  row {i+1}/{len(rows)}, pairs={len(pairs)}", flush=True)
            with open(pairs_file, "w") as f:
                for p in pairs:
                    f.write(json.dumps(p, ensure_ascii=False) + "\n")
    with open(pairs_file, "w") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"training DPO on {len(pairs)} pairs")
    if len(pairs) < 10:
        print("WARNING: too few pairs (<10), DPO skipped")
        return
    ds = Dataset.from_list(pairs)
    trainer = DPOTrainer(model=model, ref_model=None, tokenizer=tokenizer, train_dataset=ds,
        args=DPOConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_ratio=0.03, num_train_epochs=1, learning_rate=5e-5, fp16=True,
            logging_steps=25, save_strategy="no", output_dir=str(out)), callbacks=[SaveCB(out)])
    trainer.train()
    model.save_pretrained(str(out/"adapters")); tokenizer.save_pretrained(str(out/"adapters"))
    for o in out.glob("ckpt_step*"): shutil.rmtree(o, ignore_errors=True)  # training complete: drop resume checkpoints

if not marker("stage4_egopd"):
    if not HAS_CUDA:
        print("stage4_egopd: SKIP (no CUDA — run on Colab T4).")
    else:
        stage4(); mark("stage4_egopd")
else:
    print("stage4_egopd: SKIP (already done)")


In [ ]:
# @title Stage 5 — RTE SFT (reflection-on-tool-error)  [CUDA required]
def stage5():
    from unsloth import FastLanguageModel
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from datasets import Dataset
    prev = ART/"stage4_egopd"/"adapters" if (ART/"stage4_egopd"/"adapters").exists() else ART/"stage3_dtsa_sft"/"adapters"
    model, tokenizer = FastLanguageModel.from_pretrained(str(prev), max_seq_length=2048)
    model = FastLanguageModel.get_peft_model(model, r=64, lora_alpha=128, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    rows = [json.loads(l) for l in open(ART/"rte.jsonl") if l.strip()]
    def build_rte(r):
        m = DTSA_BIND.search(r["corrected"])
        name = m.group(1) if m else ""
        prompt = (f"<user>{r['query']}</user>\n<tools>{json.dumps([t.get('function',t) for t in r['tools']], ensure_ascii=False)}</tools>\n"
                  f"<calls>{r['failed']}\n<tool_error>{r['error']}</tool_error>\n<reflect/>\n" + bind_prefix(name))
        return {"text": prompt + strip_bind(r["corrected"]) + "\n"}
    ds = Dataset.from_list([build_rte(r) for r in rows])
    out = ART/"stage5_rte"
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, formatting_func=lambda x: x["text"],
        args=TrainingArguments(per_device_train_batch_size=4, gradient_accumulation_steps=4,
            warmup_ratio=0.03, num_train_epochs=1, learning_rate=1e-4, fp16=True,
            logging_steps=25, save_strategy="no", output_dir=str(out)))
    trainer.train()
    model.save_pretrained(str(out/"adapters")); tokenizer.save_pretrained(str(out/"adapters"))

if not marker("stage5_rte"):
    if not HAS_CUDA:
        print("stage5_rte: SKIP (no CUDA — run on Colab T4).")
    else:
        stage5(); mark("stage5_rte")
else:
    print("stage5_rte: SKIP (already done)")


In [ ]:
# @title Stage 6 — merge LoRA, export GGUF, eval Yantra (DTSA parser)
_Q4_DIRS = [Path("/content/stage6_yantra_q4_gguf"), Path("/content/stage6_yantra_q4"),
            ART/"stage6_yantra_q4_gguf", ART/"stage6_yantra_q4"]
def find_q4_gguf():
    for d in _Q4_DIRS:
        if d.exists():
            g = sorted(d.glob("*Q4_K_M*.gguf")) or sorted(d.glob("*.gguf"))
            if g: return g[0]
    return None

def stage6():
    from unsloth import FastLanguageModel
    import shutil
    prev = ART/"stage5_rte"/"adapters" if (ART/"stage5_rte"/"adapters").exists() else ART/"stage4_egopd"/"adapters"
    gguf = None if os.environ.get("FORCE_STAGE6") else find_q4_gguf()
    if gguf is not None:
        print(f"reusing existing GGUF: {gguf} ({gguf.stat().st_size//1024//1024}MB) "
              "(delete markers/stage6_yantra + artifacts to force re-export)")
    else:
        model, tokenizer = FastLanguageModel.from_pretrained(str(prev), max_seq_length=2048)
        # Export GGUF straight to LOCAL disk. Two rules learned the hard way:
        #  1) Drive FUSE throws [Errno 5] on multi-GB writes -> always write local,
        #     then mirror best-effort.
        #  2) save_pretrained_gguf performs its own internal 16-bit merge, so a
        #     separate save_pretrained_merged call just doubles the work.
        for d in (Path("/content/stage6_yantra_q4"), Path("/content/stage6_yantra_q4_gguf")):
            if d.exists(): shutil.rmtree(d)
        model.save_pretrained_gguf("/content/stage6_yantra_q4", tokenizer, quantization_method="q4_k_m")
        gguf = find_q4_gguf()
        assert gguf is not None, "GGUF export produced no file"
        del model, tokenizer
        import gc, torch
        gc.collect(); torch.cuda.empty_cache()
        try:
            dst = ART/"stage6_yantra_q4_gguf"
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(str(Path("/content/stage6_yantra_q4_gguf")), str(dst))
            print("GGUF mirrored to Drive")
        except Exception as e:
            print("WARNING: Drive mirror of GGUF failed (non-fatal, eval continues):", e)
    print(f"GGUF: {gguf} ({gguf.stat().st_size//1024//1024}MB)")
    # serve + eval with DTSA parser
    server = ensure_server(gguf)
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="x", timeout=120.0)
    cases = [json.loads(l) for l in open(ART/"toolace_300.jsonl") if l.strip()]
    per = []
    for i, case in enumerate(cases):
        # simple BM25 router to bind the tool (the Yantra system)
        toks = lambda s: set(re.findall(r"[a-z0-9_]+", s.lower()))
        def route(q, tools):
            best, bn = -1, None
            for t in tools:
                f = t.get("function", t); d = toks(f.get("name","")+" "+f.get("description",""))
                sc = len(toks(q) & d)/math.sqrt(len(toks(q))*len(d)+1)
                if sc > best: best, bn = sc, f.get("name")
            return bn
        bind = f'<bind tool="{route(case["query"], case["tools"])}"/>\n'
        prompt = f"<user>{case['query']}</user>\n<tools>{json.dumps([t.get('function',t) for t in case['tools']], ensure_ascii=False)}</tools>\n<calls>{bind}"
        # Stop sequences: after the answer block the model tends to ramble into
        # invented follow-up calls; a real runtime would cut generation there.
        r = client.chat.completions.create(model="yantra", messages=[{"role":"user","content":prompt}],
                                            temperature=0, max_tokens=256,
                                            stop=["\n<bind", "<tool_result>", "<user>"])
        # The bind lives in the PROMPT; grade prompt-bind + completion together,
        # else a perfectly-clean args-only completion scores as unparseable.
        m = evaluate_case(bind + (r.choices[0].message.content or ""), case, mode="dtsa")
        per.append({"id": i, **m})
        if (i+1) % 25 == 0: print(f"  eval {i+1}/{len(cases)}", flush=True)
    agg = {}
    for m in per:
        for k, v in m.items():
            if k == "id": continue
            agg[k] = agg.get(k, 0.0) + v
    n = len(cases); summary = {k: round(v/n,4) for k,v in agg.items()}
    score = pas(summary)
    res = {"summary": summary, "pas": score, "per_case": per}
    (ART/"yantra_results.json").write_text(json.dumps(res, indent=2))
    if server: server.terminate()
    print("YANTRA PAS =", score); print(json.dumps(summary, indent=2))
    return res

if not marker("stage6_yantra"):
    if not HAS_CUDA:
        print("stage6_yantra: SKIP (no CUDA — run on Colab T4).")
    else:
        stage6(); mark("stage6_yantra")
else:
    r = json.loads((ART/"yantra_results.json").read_text())
    print("stage6_yantra: SKIP (PAS =", r["pas"], ")")


In [ ]:
# @title Stage 7 — summary (in-pipeline relative beat)
def show(name):
    p = ART/f"{name}_results.json"
    if not p.exists(): return None
    return json.loads(p.read_text())
b = show("baseline"); y = show("yantra")
print("="*50)
print(f"{'metric':<16}{'baseline':>12}{'yantra':>12}")
if b and y:
    for k in ("parseable","valid_name","expected_name","exact_args","arg_key_overlap","stopped_cleanly"):
        print(f"{k:<16}{b['summary'].get(k,0):>12}{y['summary'].get(k,0):>12}")
    print(f"{'PAS':<16}{b['pas']:>12}{y['pas']:>12}")
    print("\nVERDICT:", "Yantra BEATS baseline in-pipeline" if y["pas"] > b["pas"] else "not yet")
else:
    print("run stages 2 and 6 first (need CUDA for stage 6).")


In [ ]:
# @title Stage 8 — upload trained GGUF to Hugging Face
import getpass
from huggingface_hub import login, HfApi
from pathlib import Path

token = getpass.getpass("HF token (write permission): ")
login(token)

_cands = [Path("/content/stage6_yantra_q4_gguf"), ART/"stage6_yantra_q4_gguf", ART/"stage6_yantra_q4"]
_gg = None
for _d in _cands:
    if _d.exists():
        g = sorted(_d.glob("*Q4_K_M*.gguf")) or sorted(_d.glob("*.gguf"))
        if g: _gg = g[0]; break
assert _gg is not None, "No GGUF found \u2014 run Stage 6 first"
gguf = _gg
print(f"uploading {gguf} ({gguf.stat().st_size//1024//1024}MB)")
repo = "eulogik/yantra-1b-agent"
api = HfApi()
api.create_repo(repo, exist_ok=True, private=True)
api.upload_file(path_or_fileobj=str(gguf), path_in_repo=gguf.name, repo_id=repo,
                commit_message="Yantra 1B agent \u2014 stage3 SFT + stage4 DPO + stage5 RTE (Q4_K_M GGUF)")
print(f"uploaded to https://huggingface.co/{repo}")
